In [2]:
import os

In [3]:
os.getcwd()

'd:\\Berlin_House_Price_Prediction\\Berlin_House_Price_Prediction\\research'

In [4]:
os.chdir('../')

In [5]:
%pwd

'd:\\Berlin_House_Price_Prediction\\Berlin_House_Price_Prediction'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_directory: Path
    data_path: Path

In [7]:
import os
from dataclasses import dataclass
from pathlib import Path

# Defined our schema or the variable type for our data ingestion variables which we will later import
@dataclass(frozen=True)
class DataIngestionConfig:
    root_directory: Path
    source_url: str
    local_data_file: Path
    unzip_dir:  Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_directory: Path
    unzip_data_dir:  Path
    STATUS_FILE: str
    all_schema: dict

@dataclass(frozen=True)
class DataTransformationConfig:
    root_directory: Path
    data_path: Path
    test_size: float
    random_state: int
    features: list
    target: str
    numeric_features: list
    categorical_features: list


In [8]:
from Berlin_House_Price_Prediction.constants import *
from Berlin_House_Price_Prediction.utils.common import read_yaml,create_directories
from Berlin_House_Price_Prediction.entity.config_entity import DataIngestionConfig
from Berlin_House_Price_Prediction.entity.config_entity import DataValidationConfig
from Berlin_House_Price_Prediction.entity.config_entity import DataTransformationConfig

# this class has a constructor which reads the yaml files
class ConfigurationManger:
    # Reading yaml file and create root directory
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAMS_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

        create_directories([self.config.artifacts_root])


    # # 
    # def get_data_ingestion_config(self)->DataIngestionConfig:
    #     config = self.config.data_ingestion

    #     create_directories([config.root_directory])

    #     data_ingestion_config = DataIngestionConfig(
    #         root_directory=config.root_directory,
    #         source_url=config.source_URL,
    #         local_data_file= config.local_data_file,
    #         unzip_dir= config.unzip_dir
    #     )
        
    #     return data_ingestion_config
    

    # def get_data_validation_config(self)->DataValidationConfig:
    #     config = self.config.data_validation
    #     schema = self.schema.COLUMNS

    #     create_directories([config.root_directory])

    #     data_validation_config = DataValidationConfig(
    #         root_directory= config.root_directory,
    #         unzip_data_dir= config.unzip_data_dir,
    #         STATUS_FILE = config.STATUS_FILE,
    #         all_schema= schema,
    #     )

    #     return data_validation_config
    
    def get_data_transformation_config(self)->DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_directory])

        data_transformation_config = DataTransformationConfig(
            root_directory= config.root_directory,
            data_path= config.data_path,
            test_size= config.test_size,
            random_state= config.random_state,
            features= config.features,
            target= config.target,
            numeric_features= config.numeric_features,
            categorical_features= config.categorical_features
        )

        return data_transformation_config

In [15]:
from Berlin_House_Price_Prediction.config.configuration import DataTransformationConfig
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config = config
        self.df = pd.read_csv(config.data_path)

        # Feature configuration
        self.features = config.features
        self.target = config.target
        self.numeric_features = config.numeric_features
        self.categorical_features = config.categorical_features
        self.scaler = StandardScaler()
    
    # -----------------------------
    # Cleaning
    # -----------------------------
    def clean_categorical_columns(self):
        """
        Clean categorical string columns:
        - strip whitespace
        - remove commas
        """
        categorical_cols = [col for col in self.categorical_features if col != "zipcode"]
        for col in categorical_cols:
            self.df[col] = self.df[col].str.strip().str.replace(',', '')

        # Special case cleaning
        if "energy" in self.df.columns:
            self.df["energy"] = self.df["energy"].str.replace(r"\s*offener", "", regex=True)  
    
    # -----------------------------
    # Category normalization
    # -----------------------------
    def remove_duplicate_categorical_columns(self):

        """
        Clean categorical string columns:
        - Merges synonymous heating and energy categories
        - Removes 
        """
        heating_map = {
            "Fußbodenheizung offener": "Fußbodenheizung",
            "Etagenheizung offener": "Etagenheizung",
            "Wärmepumpe offener": "Wärmepumpe",
            "Luft-/": "Wärmepumpe",
            "Wasser-": "Wärmepumpe",
        }
        energy_map = {
            'Luft-/': 'Wärmepumpe',
            'Fußbodenheizung': 'Andere',
            'Niedrigenergiehaus': 'Andere'
        }

        if "heating" in self.df.columns:
            # Merge synonymous 'heating' categories
            self.df["heating"] = self.df["heating"].replace(heating_map)

            # Merge rare 'heating' categories into 'Other'
            counts = self.df["heating"].value_counts()
            rare_categories = counts[counts < 48].index
            self.df["heating"] = self.df["heating"].replace(rare_categories, "Other")

        
        if "energy" in self.df.columns:
            # Merge synonymous 'energy' categories
            self.df["energy"] = self.df["energy"].replace(energy_map)

            # Merge rare 'energy' categories into 'Other'
            counts = self.df["energy"].value_counts()
            rare_categories = counts[counts < 20].index
            self.df["energy"] = self.df["energy"].replace(rare_categories, "Other")

    # -----------------------------
    # Outlier removal
    # -----------------------------
    def drop_outliers(self):
        """
            Drops about 2% of extreme data (100 rows)
        """
        q_low = self.df['price'].quantile(0.01)
        q_high = self.df['price'].quantile(0.99)

        self.df = self.df[(self.df['price']>q_low)&(self.df['price']<q_high)]
    # -----------------------------
    # Missing value imputation
    # -----------------------------
    def impute_missing_values(self):
        """
        Imputes categorical variables by the given probabilities
        """
        heating_top_categories = ['Zentralheizung', 'Gas', 'Fernwärme']
        energy_top_categories = ['Gas', 'Fernwärme', 'Wärmepumpe']
        probabilities_heating = [0.7, 0.15, 0.15]  
        probabilities_energy = [0.6, 0.3, 0.1]  

        self.df.loc[self.df['heating'] == 'na', 'heating'] = np.random.choice(
            heating_top_categories, 
            size=self.df[self.df['heating']=='na'].shape[0], 
            p=probabilities_heating
            )
        self.df.loc[self.df['energy'] == 'na', 'energy'] = np.random.choice(
            energy_top_categories,
            size=self.df[self.df['energy']=='na'].shape[0],
            p=probabilities_energy
            )
    # -----------------------------
    # Feature preparation
    # -----------------------------
    def prepare_features(self):
        """
        Select features and encode categorical variables
        """
        X = self.df[self.features].copy()
        X = pd.get_dummies(X, columns=self.categorical_features, drop_first=True)

        y = self.df[self.target]

        return X, y
    # -----------------------------
    # Train-test split
    # -----------------------------    
    def split_data(self, X, y):
        """
        Split dataset into train and test
        """
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        return X_train, X_test, y_train, y_test
    # -----------------------------
    # Scaling
    # -----------------------------    
    def scale_data(self, X_train, X_test):
        """
        Scale numeric features
        """
        X_train = X_train.copy()
        X_test = X_test.copy()

        X_train[self.numeric_features] = self.scaler.fit_transform(
            X_train[self.numeric_features]
        )

        X_test[self.numeric_features] = self.scaler.transform(
            X_test[self.numeric_features]
        )

        return X_train, X_test
    
    def run_transformation_pipeline(self):
        """
        Complete transformation workflow
        """

        self.clean_categorical_columns()
        self.remove_duplicate_categorical_columns()
        self.drop_outliers()
        self.impute_missing_values()

        X,y = self.prepare_features()
        X_train, X_test, y_train, y_test= self.split_data(X,y)
        X_train, X_test = self.scale_data(X_train,X_test)

        return X_train, X_test, y_train, y_test

In [16]:
try:
    config = ConfigurationManger()
    data_validation_config = config.get_data_transformation_config()
    data_validation = DataTransformation(config=data_validation_config)
    data_validation.run_transformation_pipeline()
except Exception as e:
    raise e
    

[2026-03-04 08:19:47,631: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-04 08:19:47,633: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-04 08:19:47,637: INFO: common: yaml file: <_io.TextIOWrapper name='schema.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-04 08:19:47,639: INFO: common: created directory at: artifacts]
[2026-03-04 08:19:47,643: INFO: common: created directory at: artifacts/data_transformation]
